In [ ]:
import os
import time
import pandas as pd
from relin_dataLoader import load_entity_description
from relin import RELIN
from relin_postProcess import save_summary

# can list dbpedia & lmdb together since both are inside ESBM, but faces needs to be alone (change the dataset_root appropriately)
ESBM_ROOT = "../../datasets/ESBM_benchmark_v1.2"
OUTPUT_STATS = "./outputs/stats"
DATASET_TYPE = ["dbpedia"]  # dbpedia | lmdb | faces | WikiESDataset
K = [5, 10] # 5 | 10

for dataset_type in DATASET_TYPE:
    ESBM_TYPES = ["dbpedia", "lmdb"]
    
    
    if dataset_type in ESBM_TYPES:
        DATASET_ROOT = ESBM_ROOT
    else:
        raise ValueError(f"Wrong dataset type: {dataset_type}")
    
    for k in K:
        relin = RELIN(lam=0.85, iterations=10)

        # get folder ids
        dataset_path = os.path.join(DATASET_ROOT, dataset_type + "_data")

        entity_ids = [
            d for d in os.listdir(dataset_path)
            if d.isdigit() and os.path.isdir(os.path.join(dataset_path, d))
        ]

        entity_ids = sorted(entity_ids, key=lambda x: int(x))

        print(f"Found {len(entity_ids)} entity folders under {dataset_path}")

        # track runtime
        start_total = time.time()

        # loop summary
        print(f"=== Generating Summary for {dataset_type} (k={k}) ===")
        for i in entity_ids:
            try:
                start = time.time()
                
                _, subject, features = load_entity_description(DATASET_ROOT, dataset_type, i)
                ranked = relin.rank(features)
                save_summary(dataset=DATASET_ROOT, entity_id=i, ranked=ranked, dataset_type=dataset_type, k=k)
                
                end = time.time()
            
            except Exception as e:
                print(f"[WARN] Skipping entity {i} with error: {e}")

        end_total = time.time()
        total_time = end_total - start_total

        print("\n==========================================")
        print(f"[DONE] RELIN summaries generated for all entites > ./outputs/out_summ_RELIN/{dataset_type}/")
        print("==========================================")
        print(f"Total runtime        : {total_time:.2f} seconds")

        # save runtime data
        os.makedirs(OUTPUT_STATS, exist_ok=True)
        runtime_csv = os.path.join(OUTPUT_STATS, "runtime_results.csv")

        new_row = {
            "model": "RELIN",
            "dataset": dataset_type,
            "k": k,
            "runtime (s)": total_time,
        }

        if os.path.exists(runtime_csv):
            df_existing = pd.read_csv(runtime_csv)
            # remove duplicates entry
            df_updated = (
                pd.concat([df_existing, pd.DataFrame([new_row])], ignore_index=True)
                .drop_duplicates(subset=["model", "dataset", "k"], keep="last")
            )

        else:
            df_updated = pd.DataFrame([new_row])

        df_updated.to_csv(runtime_csv, index=False)

        print("==========================================")
        print(f"[SAVE] Runtime results > {runtime_csv}\n")

Found 125 entity folders under ../../datasets/ESBM_benchmark_v1.2/dbpedia_data
=== Generating Summary for dbpedia (k=5) ===

[DONE] RELIN summaries generated for all entites > ./outputs/out_summ_RELIN/dbpedia/
Total runtime        : 1.05 seconds
[SAVE] Runtime results > ./outputs/stats/runtime_results.csv

Found 125 entity folders under ../../datasets/ESBM_benchmark_v1.2/dbpedia_data
=== Generating Summary for dbpedia (k=10) ===

[DONE] RELIN summaries generated for all entites > ./outputs/out_summ_RELIN/dbpedia/
Total runtime        : 1.07 seconds
[SAVE] Runtime results > ./outputs/stats/runtime_results.csv

